# PHASE 3: Exploratory Data Analysis (EDA)
**Traceability**
- Issue ID: #3 Exploratory Data Analysis

**External References**:
- [Carl Kirstein - Predictive Maintenance](https://www.kaggle.com/code/carlkirstein/predictive-maintenance-nasa-turbofan-regression) (Correlation Analysis)
- [Wassim Derbel - NASA Predictive Maintenance](https://www.kaggle.com/code/wassimderbel/nasa-predictive-maintenance-rul) (Trend Visualization)

## 1. Objectives
- **Diagnostic**: Analyze engine lifetime distribution to set RUL targets.
- **Diagnostic**: Visualize sensor degradation trends to identify predictive features.
- **Diagnostic**: Compute correlation matrix to detect multicollinearity.
- **Action**: Inform feature selection and engineering strategies.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# ── Plotting Config ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})
COLORS = ['#1F4E79', '#2E75B6', '#70AD47', '#FF7043', '#AB47BC']

# ── Global Config ────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load Data
df_train = pd.read_csv(PROCESSED_DIR / 'train_cleaned.csv')
sensor_cols = [c for c in df_train.columns if c.startswith('s_')]
print(f"✅ Cleaned data loaded: {df_train.shape}")

### 3.1 Diagnostic: Engine Lifetime Analysis
We need to know how long engines typically last to define "Early", "Mid", and "Late" life stages.

In [ ]:
def engine_lifetime_analysis(df):
    """Plot distribution of engine lifetimes."""
    max_cycles = df.groupby('unit_number')['time_cycles'].max()
    
    plt.figure(figsize=(10, 4))
    plt.hist(max_cycles, bins=20, color=COLORS[0], edgecolor='white')
    plt.axvline(max_cycles.mean(), color=COLORS[3], linestyle='--', label=f'Mean = {max_cycles.mean():.1f}')
    plt.axvline(max_cycles.median(), color=COLORS[2], linestyle=':', label=f'Median = {max_cycles.median():.1f}')
    plt.title('Engine Lifetime Distribution (Train FD001)')
    plt.xlabel('Cycles to Failure')
    plt.ylabel('Engine Count')
    plt.legend()
    plt.show()
    
    return max_cycles.describe()

lifetime_stats = engine_lifetime_analysis(df_train)
print("--- Engine Lifetime Stats ---")
print(lifetime_stats)

### 3.2 Diagnostic: Sensor Trends vs RUL
We calculate RUL (Max Cycle - Current Cycle) and plot sensor values. We look for monotonic trends (upward or downward) as RUL decreases.

In [ ]:
def sensor_trends_analysis(df, sensor_cols):
    """Plot sensor values vs. RUL for a subset of engines."""
    # Calculate RUL for visualization
    rul_temp = df.groupby('unit_number')['time_cycles'].max().reset_index()
    rul_temp.columns = ['unit_number', 'max_cycle']
    df_eda = df.merge(rul_temp, on='unit_number')
    df_eda['RUL'] = df_eda['max_cycle'] - df_eda['time_cycles']
    
    # Plot top sensors
    top_sensors = sensor_cols[:9] # Visualize first 9 sensors
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    
    for i, (sensor, ax) in enumerate(zip(top_sensors, axes.flatten())):
        for unit in df['unit_number'].unique()[:10]: # Plot first 10 engines
            subset = df_eda[df_eda['unit_number'] == unit].sort_values('time_cycles')
            smoothed = subset[sensor].rolling(window=10, min_periods=1).mean()
            ax.plot(subset['RUL'].values, smoothed.values, alpha=0.6, linewidth=1)
        ax.set_xlim(300, 0) # Reverse X-axis (High RUL -> 0 RUL)
        ax.set_xlabel('RUL (cycles)')
        ax.set_ylabel(sensor)
        ax.set_title(f'{sensor} vs RUL')
        
    plt.suptitle('Sensor Readings vs RUL (Reversed X-Axis)\nHealthy (Left) → Failure (Right)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

sensor_trends_analysis(df_train, sensor_cols)

### 3.3 Diagnostic: Correlation Analysis
Check for multicollinearity. If two sensors are highly correlated, they carry redundant information.

In [ ]:
def correlation_analysis(df, sensor_cols):
    """Analyze sensor correlations."""
    rul_temp = df.groupby('unit_number')['time_cycles'].max().reset_index()
    rul_temp.columns = ['unit_number', 'max_cycle']
    df_eda = df.merge(rul_temp, on='unit_number')
    df_eda['RUL'] = df_eda['max_cycle'] - df_eda['time_cycles']
    
    corr = df_eda[sensor_cols + ['RUL']].corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
    plt.title('Sensor Correlation Heatmap (including RUL)')
    plt.tight_layout()
    plt.show()
    
    return corr['RUL'].sort_values()

rul_corr = correlation_analysis(df_train, sensor_cols)
print("--- Top Correlations with RUL ---")
print(rul_corr)

**Reasoning**: We observe very high correlations (e.g., > 0.9) between certain sensors. This confirms we can use PCA or feature selection to reduce dimensionality.

In [ ]:
def pca_analysis(df, sensor_cols):
    """Perform PCA to assess global variance."""
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(df[sensor_cols])
    
    pca = PCA(n_components=5)
    pca.fit(X_scaled)
    
    explained_variance = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)
    
    plt.figure(figsize=(8, 4))
    plt.bar(range(1, 6), explained_variance * 100, color=COLORS[0], alpha=0.8, label='Individual')
    plt.plot(range(1, 6), cumulative_variance * 100, 'r-o', label='Cumulative')
    plt.xlabel('PCA Component')
    plt.ylabel('Explained Variance %')
    plt.title(f'PCA Variance — PC1 explains {explained_variance[0]*100:.1f}%')
    plt.legend()
    plt.tight_layout()
    plt.show()

pca_analysis(df_train, sensor_cols)